# Complete Analysis Pipeline - Swiss Political Parties on Instagram & TikTok

**Group**: Nick Eichmann, Marc Eggenberger, Sarah Häusermann, David Rothschild

This notebook runs ALL analysis scripts from `1_Processing/2_Analysis/` in sequence.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavidSimonRothschild/Introduction-to-Computational-Media-Research/blob/main/complete_analysis_colab.ipynb)

## Setup: Clone Repository & Install Dependencies

In [ ]:
import os
import sys

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    !rm -rf repo
    !git clone https://github.com/DavidSimonRothschild/Introduction-to-Computational-Media-Research.git repo
    os.chdir('repo')
else:
    print("Running locally")

PROJECT_ROOT = os.getcwd()
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Install dependencies
!pip install -q pandas numpy scipy statsmodels nltk networkx matplotlib

# Download NLTK data for tokenization
import nltk
nltk.download('punkt', quiet=True)
print("Dependencies installed")

---
## Part 1: Sentiment Analysis

Runs sentiment analysis on all Instagram and TikTok posts using German sentiment lexicons.

In [ ]:
print("=" * 60)
print("PART 1: SENTIMENT ANALYSIS")
print("=" * 60)

# Run sentiment analysis for TikTok
print("\n--- TikTok Sentiment Analysis ---")
!python 1_Processing/2_Analysis/1_Caption_Sentiment/sentiment_analysis_tiktok.py

# Run sentiment analysis for Instagram
print("\n--- Instagram Sentiment Analysis ---")
!python 1_Processing/2_Analysis/1_Caption_Sentiment/sentiment_analysis_instagram.py

print("\n✓ Sentiment analysis completed")

---
## Part 2: Label Voting Topics

Labels posts as voting-related (E-ID, Eigenmietwert) or non-voting.

In [ ]:
print("=" * 60)
print("PART 2: LABEL VOTING TOPICS")
print("=" * 60)

!python 1_Processing/2_Analysis/3_Label_Posts_Voting_Topic/label_posts.py

print("\n✓ Voting topic labeling completed")

---
## Part 3: Calculate Engagement Scores

Calculates mean-centered engagement scores for each party.

In [ ]:
print("=" * 60)
print("PART 3: ENGAGEMENT SCORES")
print("=" * 60)

# Instagram engagement scores
print("\n--- Instagram Engagement Scores ---")
!python 1_Processing/2_Analysis/4_Engagement_Score/1_Engagement_score_Instagram.py

# TikTok engagement scores
print("\n--- TikTok Engagement Scores ---")
!python 1_Processing/2_Analysis/4_Engagement_Score/2_Engagement_score_Tiktok.py

print("\n✓ Engagement score calculation completed")

---
## Part 4: Network Analysis

Analyzes cross-party mentions and creates network visualizations.

In [ ]:
print("=" * 60)
print("PART 4: NETWORK ANALYSIS")
print("=" * 60)

# Instagram network analysis
print("\n--- Instagram Network Analysis ---")
!python "1_Processing/2_Analysis/2_Network Analysis/network_analysis_party_mentions_instagram.py"

# TikTok network analysis
print("\n--- TikTok Network Analysis ---")
!python "1_Processing/2_Analysis/2_Network Analysis/network_analysis_party_mentions_tiktok.py"

print("\n✓ Network analysis completed")

---
## Part 5: Hypothesis Testing

Now that all data is processed, run the hypothesis tests.

In [ ]:
import pandas as pd
import numpy as np
import glob
from scipy.stats import mannwhitneyu, spearmanr
import statsmodels.api as sm

# Load data
def load_platform(path):
    files = glob.glob(os.path.join(path, "*.csv"))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["party_file"] = os.path.basename(f)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

INSTAGRAM_FOLDER = os.path.join(PROJECT_ROOT, "A_Data", "2_Instagram", "2_CLEAN")
TIKTOK_FOLDER = os.path.join(PROJECT_ROOT, "A_Data", "1_Tiktok", "2_CLEAN")

tiktok = load_platform(TIKTOK_FOLDER)
instagram = load_platform(INSTAGRAM_FOLDER)

tiktok["platform"] = "tiktok"
instagram["platform"] = "instagram"

df = pd.concat([tiktok, instagram], ignore_index=True)
print(f"Total posts loaded: {len(df)}")
print(f"Instagram: {len(instagram)}, TikTok: {len(tiktok)}")

### H1: Voting Issues and Engagement

In [ ]:
print("=" * 60)
print("H1: VOTING ISSUES AND ENGAGEMENT")
print("=" * 60)

df_h1 = df.dropna(subset=["engagement_score", "voting.topic"])
df_h1["is_any_voting"] = (df_h1["voting.topic"] > 0).astype(int)

voting_all = df_h1.loc[df_h1.is_any_voting == 1, "engagement_score"]
non_voting = df_h1.loc[df_h1.is_any_voting == 0, "engagement_score"]

u, pval = mannwhitneyu(voting_all, non_voting, alternative="greater")

print(f"n voting: {len(voting_all)}, n non-voting: {len(non_voting)}")
print(f"Mann-Whitney U p-value: {pval:.4f}")
print(f"Median engagement – voting: {voting_all.median():.2f}")
print(f"Median engagement – non-voting: {non_voting.median():.2f}")

if pval > 0.95:
    print("\n>>> RESULT: H1 REJECTED - Voting posts have LOWER engagement!")
else:
    print("\n>>> RESULT: H1 supported" if pval < 0.05 else ">>> RESULT: No significant difference")

### H2: Temporal Proximity

In [ ]:
print("=" * 60)
print("H2: TEMPORAL PROXIMITY TO VOTING DAY")
print("=" * 60)

VOTING_DAY = pd.Timestamp("2025-09-28")

df_h2 = df.dropna(subset=["engagement_score", "voting.topic", "data.createTime"])
df_h2["post_date"] = pd.to_datetime(df_h2["data.createTime"])
df_h2["days_to_vote"] = (VOTING_DAY - df_h2["post_date"]).dt.days

pre_vote = df_h2[df_h2["days_to_vote"] >= 0]

rho, pval = spearmanr(pre_vote["days_to_vote"], pre_vote["engagement_score"], alternative="less")

print(f"Spearman rho: {rho:.3f}")
print(f"p-value: {pval:.4f}")

if pval > 0.95:
    print("\n>>> RESULT: H2 REJECTED - Posts FURTHER from vote have higher engagement!")
elif pval < 0.05:
    print("\n>>> RESULT: H2 supported")
else:
    print("\n>>> RESULT: No significant relationship")

### H3: Negativity and Engagement

In [ ]:
print("=" * 60)
print("H3: NEGATIVITY AND ENGAGEMENT")
print("=" * 60)

SENTIMENT_COL = "sentiment_rulebased"
Y_COL = "engagement_score"

results_h3 = []

for platform, folder in [("Instagram", INSTAGRAM_FOLDER), ("TikTok", TIKTOK_FOLDER)]:
    print(f"\n--- {platform} ---")
    
    pooled_rows = []
    
    for csv_path in sorted(glob.glob(os.path.join(folder, "*.csv"))):
        party_df = pd.read_csv(csv_path)
        
        if SENTIMENT_COL not in party_df.columns or Y_COL not in party_df.columns:
            continue
        
        d = party_df[[SENTIMENT_COL, Y_COL]].copy()
        d[SENTIMENT_COL] = pd.to_numeric(d[SENTIMENT_COL], errors="coerce")
        d[Y_COL] = pd.to_numeric(d[Y_COL], errors="coerce")
        d = d.dropna()
        
        if len(d) >= 10:
            pooled_rows.append(d)
    
    if pooled_rows:
        pooled_df = pd.concat(pooled_rows, ignore_index=True)
        Xp = sm.add_constant(pooled_df[SENTIMENT_COL])
        yp = pooled_df[Y_COL]
        pooled_model = sm.OLS(yp, Xp).fit(cov_type="HC3")
        
        print(f"Pooled: n={int(pooled_model.nobs)}, beta={pooled_model.params[SENTIMENT_COL]:.3f}, p={pooled_model.pvalues[SENTIMENT_COL]:.3f}")

print("\n>>> RESULT: H3 REJECTED - No systematic relationship between sentiment and engagement")

### H4: Ideological Distance and Engagement

In [ ]:
print("=" * 60)
print("H4: IDEOLOGICAL DISTANCE AND ENGAGEMENT")
print("=" * 60)

NETWORK_INSTAGRAM = os.path.join(PROJECT_ROOT, "1_Processing", "2_Analysis", "2_Network Analysis", "party_mentions_edges_instagram.csv")
NETWORK_TIKTOK = os.path.join(PROJECT_ROOT, "1_Processing", "2_Analysis", "2_Network Analysis", "party_mentions_edges_tiktok.csv")

IDEOLOGY = {
    "JUSO": 4.4, "SP": 4.4,
    "Junge Grüne": 5.6, "Grüne": 5.6,
    "EVP": 30.9, "Junge EVP": 30.9,
    "Junge GLP": 53.8, "GLP": 53.8,
    "Junge Mitte": 62.4, "Mitte": 62.4,
    "JF": 79.8, "FDP": 79.8,
    "JSVP": 91.9, "SVP": 91.9
}

ig = pd.read_csv(NETWORK_INSTAGRAM)
tt = pd.read_csv(NETWORK_TIKTOK)

ig["platform"] = "instagram"
tt["platform"] = "tiktok"

edges = pd.concat([ig, tt], ignore_index=True)

edges["ideo_dist"] = edges.apply(
    lambda r: abs(IDEOLOGY.get(r.source_party, 50) - IDEOLOGY.get(r.target_party, 50)),
    axis=1
)

rho, pval = spearmanr(edges["ideo_dist"], edges["mean_engagement"], alternative="greater")

print(f"Spearman rho: {rho:.3f}")
print(f"p-value: {pval:.3f}")

print("\n>>> RESULT: H4 REJECTED - No correlation between ideological distance and engagement")

---
## Summary

| Hypothesis | Result |
|------------|--------|
| **H1**: Voting posts → higher engagement | ❌ REJECTED (opposite effect) |
| **H2**: Closer to vote → higher engagement | ❌ REJECTED |
| **H3**: Negativity → higher engagement | ❌ REJECTED |
| **H4**: Ideological distance → higher engagement | ❌ REJECTED |